# HUGO Gene Nomenclature Committee (HGNC) — Data Ingestion

**HGNC** is the international authority responsible for approving unique, standardised symbols and names for every human gene. Maintained at the European Bioinformatics Institute (EBI), it is one of the ELIXIR Core Data Resources and serves as the canonical reference for human gene identity across genomics, proteomics, and clinical databases worldwide.

Every approved entry carries a stable **HGNC ID** that persists even when a gene symbol is updated, making it invaluable as a cross-database foreign key.

Key data types provided by HGNC:

| Field | Description |
|---|---|
| `hgnc_id` | Stable primary accession, e.g. `HGNC:5` |
| `symbol` | Approved gene symbol, e.g. `A1BG` |
| `name` | Full approved gene name |
| `locus_group` | Broad category: `protein-coding gene`, `non-coding RNA`, `pseudogene`, etc. |
| `locus_type` | Fine-grained locus classification |
| `status` | `Approved`, `Entry Withdrawn`, etc. |
| `location` | Cytogenetic band, e.g. `19q13.43` |
| `entrez_id` | NCBI Entrez Gene accession |
| `ensembl_gene_id` | Ensembl stable gene ID |
| `uniprot_ids` | Corresponding UniProtKB accession(s) |
| `omim_id` | OMIM disease database accession(s) |
| `date_approved_reserved` | Date the symbol was first approved |
| `date_modified` | Date the entry was last updated |

**REST API base:** `https://rest.genenames.org`

**Bulk download:** `https://ftp.ebi.ac.uk/pub/databases/genenames/hgnc/tsv/hgnc_complete_set.txt`

**Reference:** Tweedie et al. (2021), *Nucleic Acids Research*, HGNC: The HUGO Gene Nomenclature Committee. https://doi.org/10.1093/nar/gkaa980

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to the HGNC REST API and confirm access via `/info`
    * [x] Download the complete HGNC dataset via bulk FTP TSV
    * [x] Parse into a Polars DataFrame with correct dtypes
    * [x] Fetch gene info for a specific symbol via the search API
    * [x] Save data to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise entry counts by locus group and locus type
    * [ ] Inspect completeness of cross-reference columns (Ensembl, UniProt, OMIM)
    * [ ] Identify withdrawn entries and handle them appropriately
* [ ] **Analysis**
    * [ ] Analyse symbol approval trends over time
    * [ ] Explore the distribution of genes across chromosomes and cytobands
    * [ ] Cross-reference with Ensembl/UniProt to measure annotation coverage
* [ ] **Visualization**
    * [ ] Bar chart of gene counts per locus group
    * [ ] Timeline of approvals per year coloured by locus group
    * [ ] Chromosome ideogram showing gene density
* [ ] **Statistical analysis**
    * [ ] Discuss uncertainty in gene count estimates (withdrawn entries, symbol merges)
    * [ ] Multiple hypothesis correction considerations for genome-wide symbol queries
    * [ ] Statistical framework for assessing cross-database ID mapping completeness

## 1. Ingest Data

### 1.1 Connect to the HGNC REST API

In [ ]:
HGNC_REST_BASE = "https://rest.genenames.org"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)  # create data/ directory if it does not exist

# The HGNC REST API requires Accept: application/json — without this header
# the server returns XML by default.
HEADERS = {"Accept": "application/json"}


def hgnc_get(endpoint: str, params: dict = None) -> dict:
    """
    Send a GET request to the HGNC REST API and return parsed JSON.

    Parameters
    ----------
    endpoint : str
        Path relative to HGNC_REST_BASE, e.g. ``"info"`` or
        ``"search/symbol/BRCA1"``.
    params : dict, optional
        Additional query-string parameters.

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    url = f"{HGNC_REST_BASE}/{endpoint}"
    resp = requests.get(url, headers=HEADERS, params=params or {}, timeout=30)
    resp.raise_for_status()
    time.sleep(0.3)  # polite delay — EBI fair-use policy
    return resp.json()


# ── Connectivity check: /info returns database-level metadata ────────────────
info = hgnc_get("info")

# The response wraps everything under a "response" key
response_meta = info.get("response", {})

print("HGNC REST API — /info")
print("=" * 50)
print(f"  Last modified  : {response_meta.get('lastModified', 'N/A')}")
print(f"  Number of docs : {response_meta.get('numFound', 'N/A')}")

# "searchableFields" lists every field we can query via /search
searchable = response_meta.get("searchableFields", [])
print(f"  Searchable fields ({len(searchable)}): {', '.join(searchable[:10])} ...")

# "storedFields" lists every field returned in a full fetch
stored = response_meta.get("storedFields", [])
print(f"  Stored fields   ({len(stored)}): {', '.join(stored[:10])} ...")

### 1.2 Download the Complete HGNC Dataset (Bulk TSV)

In [ ]:
BULK_URL = (
    "https://ftp.ebi.ac.uk/pub/databases/genenames/hgnc/tsv/hgnc_complete_set.txt"
)
BULK_CACHE = DATA_DIR / "hgnc_complete_set.tsv"


def download_hgnc_bulk(url: str, cache_path: Path) -> Path:
    """
    Download the HGNC complete set TSV from the EBI FTP mirror.

    The file is ~10 MB and contains one row per HGNC entry with all
    approved symbols, cross-references, and metadata. We stream the
    response to disk to avoid holding the entire payload in memory.

    Parameters
    ----------
    url : str
        Direct URL to the TSV file on the EBI FTP server.
    cache_path : Path
        Local path to write (or read from if already cached).

    Returns
    -------
    Path
        Path to the downloaded (or cached) TSV file.
    """
    if cache_path.exists():
        # File already on disk — skip the download entirely
        print(f"Cache hit: {cache_path}  ({cache_path.stat().st_size / 1_048_576:.1f} MB)")
        return cache_path

    print(f"Downloading: {url}")
    with requests.get(url, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        # Write in 64 KB chunks to keep memory usage flat
        with cache_path.open("wb") as fh:
            for chunk in resp.iter_content(chunk_size=65_536):
                fh.write(chunk)

    print(f"Saved to: {cache_path}  ({cache_path.stat().st_size / 1_048_576:.1f} MB)")
    return cache_path


bulk_path = download_hgnc_bulk(BULK_URL, BULK_CACHE)

### 1.3 Parse into a Polars DataFrame with Correct Dtypes

In [ ]:
# Column subsets and their intended Polars dtypes.
# The TSV contains ~50 columns; we retain the most analytically useful ones.
# Many cross-reference columns hold pipe-separated lists of IDs (stored as Utf8).

# Columns to parse as calendar dates (format: YYYY-MM-DD in the source file)
DATE_COLS = [
    "date_approved_reserved",
    "date_symbol_changed",
    "date_name_changed",
    "date_modified",
]

# Integer columns (NCBI/OMIM IDs arrive as bare numbers but have no arithmetic meaning)
INT_COLS = ["entrez_id"]

# Categorical columns — low cardinality, benefit from dictionary encoding
CAT_COLS = ["status", "locus_group", "locus_type"]


def parse_hgnc_tsv(path: Path) -> pl.DataFrame:
    """
    Read the HGNC complete-set TSV and cast columns to appropriate dtypes.

    The raw file uses a tab separator, encodes missing values as empty strings,
    and stores multi-valued cross-references (e.g. UniProt, OMIM) as
    pipe-separated strings within a single cell.

    Parameters
    ----------
    path : Path
        Path to the downloaded ``hgnc_complete_set.tsv`` file.

    Returns
    -------
    pl.DataFrame
        Cleaned DataFrame with:
        - date columns cast to ``pl.Date``
        - integer ID columns cast to ``pl.Int64`` (nulls preserved)
        - low-cardinality columns cast to ``pl.Categorical``
        - all other columns remaining as ``pl.Utf8``
    """
    # Read everything as strings first — avoids Polars mis-inferring mixed columns
    df = pl.read_csv(
        path,
        separator="\t",
        infer_schema_length=0,  # read all columns as Utf8 initially
        null_values=["", '""'],  # HGNC uses empty string for missing values
        quote_char=None,         # the file is not quoted; avoid mis-parsing
        truncate_ragged_lines=True,  # a small number of rows have trailing tabs
    )

    # ── Cast date columns ────────────────────────────────────────────────────
    # Dates in this file look like "2013-06-27" — straightforward ISO-8601
    date_exprs = [
        pl.col(c).str.to_date("%Y-%m-%d", strict=False).alias(c)
        for c in DATE_COLS
        if c in df.columns
    ]

    # ── Cast integer columns ─────────────────────────────────────────────────
    # entrez_id is stored as a plain integer string; cast with strict=False so
    # malformed cells become null rather than raising an error
    int_exprs = [
        pl.col(c).cast(pl.Int64, strict=False).alias(c)
        for c in INT_COLS
        if c in df.columns
    ]

    # ── Cast categorical columns ─────────────────────────────────────────────
    cat_exprs = [
        pl.col(c).cast(pl.Categorical).alias(c)
        for c in CAT_COLS
        if c in df.columns
    ]

    # Apply all casts in a single .with_columns() pass for efficiency
    df = df.with_columns(date_exprs + int_exprs + cat_exprs)

    return df


hgnc_df = parse_hgnc_tsv(bulk_path)

print(f"Shape  : {hgnc_df.shape[0]:,} rows × {hgnc_df.shape[1]} columns")
print(f"\nSchema (first 20 columns):")
for col, dtype in list(hgnc_df.schema.items())[:20]:
    print(f"  {col:<40} {dtype}")
print()
hgnc_df.head(5)